First a little test to see how N and epochs affects MSE. I want to use a smaller N and epoch value to keep runtime down, so it's good to know what a larger N and epoch will do to the MSE. I do the simplest case with GD and Sigmoid. It's kind of a pre-test for part b, that I will use for the rest of the project. 

In [ ]:
from pathlib import Path
import sys


here = Path.cwd()
candidates = [here] + list(here.parents)
for p in candidates:
    if (p / "Code").is_dir():
        sys.path.insert(0, str(p))
        break
else:
    raise RuntimeError("Couldn't find a 'Code' folder in this project.")

import autograd.numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from math import ceil
from Code.data import runge_function, make_data
from Code.comparison import run_comparison
from Code.scheduler import Constant
from Code.ffnn2 import NeuralNetwork as FFNN
from Code.cost import CostOLS

 
output_nodes = 1
batches_gd = 1 
lam = 0.0
hidden_nodes1 = 50
hidden_nodes2 = 100

(seed_fixed,
 rng_fixed,
 X_fixed_test,
 y_fixed_test,
 N,
 rng,
 x,
 noise,
 y,
 X_train,
 X_val,
 y_train,
 y_val,
 scaler,
 X_train_scaled,
 X_val_scaled,
 X_fixed_test_scaled) = make_data(seed_fixed=42, N=200, noise_std=0.1, test_N=2000)



# Architecture 1: One Hidden Layer (50 nodes) 
layer_sizes_1H = [hidden_nodes1, output_nodes]

# architecture 
ARCH_FOR_TEST = layer_sizes_1H   

# comparison setup
N_list = [30, 60, 100, 200, 500, 1000, 2000]
eta_vals = np.logspace(-5, -1, 5) # Sweep 5 values
epochs_sweep = [100, 500, 1000]  # keep this reasonable to avoid long runtimes



# function to make training data
def make_train_xy(N, sigma=0.1, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, size=(N, 1))
    y = runge_function(X) + rng.normal(0, sigma, size=(N, 1))
    return X, y


rows = []

# GD + Sigmoid sweep over N, eta, epochs 
print("\n=== GD + Sigmoid sweep over N, eta, epochs (MSE on fixed test set) ===")
for N in N_list:
    # build a new train set of size N
    X_tr, y_tr = make_train_xy(N=N, sigma=0.1, seed=42)

    # scale data
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_test_fixed_s = scaler.transform(X_fixed_test)
    # sweep over epochs and eta
    for epochs in epochs_sweep:            
        for eta in eta_vals:
            # Full-batch GD: batches=1
            scores, nn = run_comparison(
                Constant,                 # optimizer class (GD)
                X_tr_s, y_tr,
                network_input_size=1,     # single feature
                layer_output_sizes=ARCH_FOR_TEST,
                epochs=int(epochs),       
                lam=lam,
                eta=float(eta),
                batches=1                 
            )

            # evaluate on fixed test set
            y_pred = nn.predict(X_test_fixed_s)
            mse = float(CostOLS(y_pred, y_fixed_test))

            rows.append(dict(N=N, epochs=int(epochs), eta=float(eta), mse=mse))
            print(f"N={N:5d} | epochs={epochs:4d} | eta={eta:.1e} -> MSE={mse:.6f}")

# Create DataFrame
gd_sigmoid_df = pd.DataFrame(rows).sort_values(["N", "epochs", "eta"]).reset_index(drop=True)
display(gd_sigmoid_df)

# Top 10
top_overall = gd_sigmoid_df.nsmallest(10, "mse").reset_index(drop=True)
print("Top 10 (lowest MSE overall):")
print(top_overall)

# Best (eta, epochs) per N
best_per_N = (
    gd_sigmoid_df.loc[gd_sigmoid_df.groupby("N")["mse"].idxmin()]
    .sort_values("mse")
    .reset_index(drop=True)
)
print("\nBest (eta, epochs) per N (ranked by MSE):")
print(best_per_N)

# Top 3 per N
topk_per_N = (
    gd_sigmoid_df.sort_values(["N", "mse"])
    .groupby("N", as_index=False)
    .head(3)
    .reset_index(drop=True)
)
print("\nTop 3 per N:")
print(topk_per_N)

# Pretty print ranked list
ranked_list = list(top_overall.itertuples(index=False, name=None))
print("\nRanked list (overall):")
for i, (N, epochs, eta, mse) in enumerate(ranked_list, 1):
    print(f"{i:2d}. N={N:<5d}  epochs={epochs:<4d}  eta={eta:.1e}  MSE={mse:.6f}")


=== GD + Sigmoid sweep over N, eta, epochs (MSE on FIXED test set) ===
N=   30 | epochs= 100 | eta=1.0e-05 -> MSE=0.193293
N=   30 | epochs= 100 | eta=1.0e-04 -> MSE=0.157054
N=   30 | epochs= 100 | eta=1.0e-03 -> MSE=0.095250
N=   30 | epochs= 100 | eta=1.0e-02 -> MSE=0.094079
N=   30 | epochs= 100 | eta=1.0e-01 -> MSE=0.094068
N=   30 | epochs= 500 | eta=1.0e-05 -> MSE=0.175099
N=   30 | epochs= 500 | eta=1.0e-04 -> MSE=0.103390
N=   30 | epochs= 500 | eta=1.0e-03 -> MSE=0.094079
N=   30 | epochs= 500 | eta=1.0e-02 -> MSE=0.094076
N=   30 | epochs= 500 | eta=1.0e-01 -> MSE=0.094036
N=   30 | epochs=1000 | eta=1.0e-05 -> MSE=0.157093
N=   30 | epochs=1000 | eta=1.0e-04 -> MSE=0.095304
N=   30 | epochs=1000 | eta=1.0e-03 -> MSE=0.094079
N=   30 | epochs=1000 | eta=1.0e-02 -> MSE=0.094072
N=   30 | epochs=1000 | eta=1.0e-01 -> MSE=0.094065
N=   60 | epochs= 100 | eta=1.0e-05 -> MSE=0.193644
N=   60 | epochs= 100 | eta=1.0e-04 -> MSE=0.159538
N=   60 | epochs= 100 | eta=1.0e-03 -> MSE=0

,N,epochs,eta,mse
0,30,100,0.00001,0.193293
1,30,100,0.00010,0.157054
2,30,100,0.00100,0.095250
3,30,100,0.01000,0.094079
4,30,100,0.10000,0.094068
...,...,...,...,...
100,2000,1000,0.00001,0.093683
101,2000,1000,0.00010,0.093683
102,2000,1000,0.00100,0.093684
103,2000,1000,0.01000,0.093854


Top 10 (lowest MSE overall):
      N  epochs      eta       mse
0  2000    1000  0.10000  0.027101
1  2000     500  0.10000  0.093567
2  2000     100  0.00100  0.093683
3  2000    1000  0.00010  0.093683
4  2000     500  0.00010  0.093683
5  2000    1000  0.00001  0.093683
6  2000     100  0.00010  0.093683
7  2000     500  0.00100  0.093683
8  2000    1000  0.00100  0.093684
9   500     500  0.01000  0.093686

Best (eta, epochs) per N (ranked by MSE):
      N  epochs     eta       mse
0  2000    1000  0.1000  0.027101
1   500     500  0.0100  0.093686
2   200     500  0.0100  0.093687
3   100    1000  0.0001  0.093708
4  1000     500  0.0100  0.093732
5    30     500  0.1000  0.094036
6    60    1000  0.1000  0.095298

Top 3 per N:
       N  epochs     eta       mse
0     30     500  0.1000  0.094036
1     30    1000  0.1000  0.094065
2     30     100  0.1000  0.094068
3     60    1000  0.1000  0.095298
4     60     500  0.1000  0.095308
5     60     100  0.1000  0.095322
6    100    